# Conditional GAN on MNIST

A teaching reference that extends `dcgan_mnist.py` with class conditions. Read [the lecture notes](conditional_gan.md) alongside this notebook.

**Goal:** request a digit with a label and vary its appearance with random noise. Both generator and discriminator receive the label. This is a conditional **DCGAN** using 32 × 32 grayscale images.

Run cells in order. A GPU is useful for full training; the first MNIST download needs internet access. Set `SMOKE_TEST = True` for a two-batch pipeline run, which is too short to learn recognizable digits.

In [ ]:
# Uncomment if these packages are missing in your notebook environment.
# %pip install torch torchvision matplotlib

In [ ]:
from pathlib import Path
import random
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM, NUM_CLASSES, IMG_SIZE = 100, 10, 32
BATCH_SIZE, EPOCHS, LR = 128, 30, 2e-4
SMOKE_TEST = False
OUTPUT_DIR = Path("conditional_gan_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 1. Conditional networks

`class_labels` contains integer digit IDs. The generator concatenates their one-hot encoding with noise. The discriminator broadcasts that encoding over the image and concatenates it as extra channels.

Generator: `(B, 110, 1, 1) → (B, 256, 4, 4) → (B, 128, 8, 8) → (B, 64, 16, 16) → (B, 1, 32, 32)`.

Discriminator input: one image channel + ten condition channels. Its output is one **logit** per pair, not ten class scores. We use `BCEWithLogitsLoss`, so there is no output sigmoid.

In [ ]:
from tkinter.ttk import Label


class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim + num_classes, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
            nn.Tanh(),  # Match normalized real pixels in [-1, 1].
        )

    def forward(self, noise, class_labels):
        condition = F.one_hot(class_labels, self.num_classes).to(dtype=noise.dtype)
        combined = torch.cat((noise, condition), dim=1)
        return self.net(combined[:, :, None, None])


class Discriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.net = nn.Sequential(
            # input dimension (1 + num_classes) x 32 x 32 -> output dimension 64 x 16 x 16
            nn.Conv2d(in_channels=1 + num_classes, out_channels=64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input dimension 64 x 16 x 16 -> output dimension 128 x 8 x 8
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            # input dimension 128 x 8 x 8 -> output dimension 256 x 4 x 4
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            # input dimension 256 x 4 x 4 -> output dimension 1 x 1 x 1
            # A single scalar output (logit) for each image in the batch, indicating real/fake.
            nn.Conv2d(in_channels=256, out_channels=1, kernel_size=4, stride=1, padding=0, bias=False),
        )

    def forward(self, images, class_labels):
        condition = F.one_hot(class_labels, self.num_classes).to(dtype=images.dtype)
        # Expand the condition tensor to match the spatial dimensions of the images.
        # The condition tensor is reshaped to have dimensions (batch_size, num_classes, 1, 1) 
        # and then expanded to (batch_size, num_classes, height, width) to match the images.
        maps = condition[:, :, None, None].expand(-1, -1, *images.shape[-2:])
        # Concatenate the images and the condition maps along the channel dimension (dim=1).
        # For e.g. 
        # Image:        (64,  1, 32, 32)
        # Label maps:   (64, 10, 32, 32)
        # Combined:     (64, 11, 32, 32)
        return self.net(torch.cat((images, maps), dim=1)).flatten(1)


def weights_init(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, 0.0, 0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)

## 2. One alternating training step

The class condition (0–9) is distinct from the adversarial target (real=1, fake=0). Real and generated pairs use the same batch of class conditions.

Step A detaches generated images. Step B freezes discriminator parameters and its BatchNorm statistics but keeps the gradient path through the discriminator to the generator. Evaluation mode does **not** disable autograd.

In [ ]:
criterion = nn.BCEWithLogitsLoss()


def train_step(generator, discriminator, opt_g, opt_d, real_images, class_labels):
    batch_size = real_images.size(0)
    device = real_images.device
    generator.train()
    discriminator.train()

    # A: learn to accept real pairs and reject generated pairs.
    opt_g.zero_grad(set_to_none=True)
    opt_d.zero_grad(set_to_none=True)
    noise = torch.randn(batch_size, LATENT_DIM, device=device)
    # Because of the detach() here, the generator weights are not updated in this step.
    fake_images = generator(noise, class_labels).detach()
    real_logits = discriminator(real_images, class_labels)
    fake_logits = discriminator(fake_images, class_labels)
    d_loss = (criterion(real_logits, torch.ones_like(real_logits))
              + criterion(fake_logits, torch.zeros_like(fake_logits)))
    d_loss.backward()
    opt_d.step()
    opt_d.zero_grad(set_to_none=True)

    # B: make generated pairs receive the adversarial target 'real'.
    # We freeze the discriminator weights for this step, so that only the generator is updated.
    discriminator.requires_grad_(False)
    discriminator.eval()  # Freeze BatchNorm running statistics for this step.
    try:
        noise = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = generator(noise, class_labels)  # Do not detach here.
        fake_logits = discriminator(fake_images, class_labels)
        g_loss = criterion(fake_logits, torch.ones_like(fake_logits))
        opt_g.zero_grad(set_to_none=True)
        g_loss.backward()
        opt_g.step()
    finally:
        discriminator.requires_grad_(True)
        discriminator.train()
    return d_loss.item(), g_loss.item()

## 3. Synthetic smoke check (no download required)

Check image/score shapes, output range, finite losses, and parameter updates using temporary models. These synthetic images are only for checking the code. Fresh models are created for MNIST training afterward.

In [ ]:
def smoke_check():
    test_g = Generator().to(DEVICE).apply(weights_init)
    test_d = Discriminator().to(DEVICE).apply(weights_init)
    test_opt_g = torch.optim.Adam(test_g.parameters(), lr=LR, betas=(0.5, 0.999))
    test_opt_d = torch.optim.Adam(test_d.parameters(), lr=LR, betas=(0.5, 0.999))
    labels = torch.arange(4, device=DEVICE, dtype=torch.long)
    with torch.no_grad():
        images = test_g(torch.randn(4, LATENT_DIM, device=DEVICE), labels)
        assert images.shape == (4, 1, IMG_SIZE, IMG_SIZE)
        assert images.min() >= -1 and images.max() <= 1
        assert test_d(images, labels).shape == (4, 1)
    g_before = next(test_g.parameters()).detach().clone()
    d_before = next(test_d.parameters()).detach().clone()
    real = torch.rand(4, 1, IMG_SIZE, IMG_SIZE, device=DEVICE) * 2 - 1
    losses = train_step(test_g, test_d, test_opt_g, test_opt_d, real, labels)
    assert torch.isfinite(torch.tensor(losses)).all()
    assert not torch.equal(g_before, next(test_g.parameters()))
    assert not torch.equal(d_before, next(test_d.parameters()))
    assert all(p.grad is None for p in test_d.parameters())
    print("Smoke check passed. D/G losses:", losses)

smoke_check()

## 4. MNIST data

Resize 28 × 28 to 32 × 32 and map pixels from [0, 1] to [-1, 1] to match `Tanh`. Keep the class labels that the unconditional script discarded. `num_workers=0` keeps this example easy to run in notebook environments.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
training_data = Subset(dataset, range(2 * BATCH_SIZE)) if SMOKE_TEST else dataset
train_loader = DataLoader(
    training_data, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=(DEVICE.type == "cuda"),
)
print(f"Training images: {len(training_data):,}")

## 5. Initialize models and fixed samples

Each sample-grid row requests a digit from 0 to 9. Each column uses the same noise across digit classes; the whole grid stays fixed across epochs. Equal noise does not guarantee equal handwriting style across classes.

In [ ]:
torch.manual_seed(SEED)
G = Generator().to(DEVICE).apply(weights_init)
D = Discriminator().to(DEVICE).apply(weights_init)
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
g_losses, d_losses = [], []
SAMPLES_PER_CLASS = 8
fixed_labels = torch.arange(NUM_CLASSES, device=DEVICE).repeat_interleave(SAMPLES_PER_CLASS)
fixed_noise = torch.randn(SAMPLES_PER_CLASS, LATENT_DIM, device=DEVICE).repeat(NUM_CLASSES, 1)


@torch.no_grad()
def generate_images(noise, labels):
    was_training = G.training
    G.eval()
    try:
        return ((G(noise, labels).cpu() + 1) / 2).clamp(0, 1)
    finally:
        G.train(was_training)


def show_class_grid(epoch):
    images = generate_images(fixed_noise, fixed_labels)
    save_image(images, OUTPUT_DIR / f"epoch_{epoch:03d}.png", nrow=SAMPLES_PER_CLASS)
    fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(8, 10))
    for row in range(NUM_CLASSES):
        for col in range(SAMPLES_PER_CLASS):
            ax = axes[row, col]
            ax.imshow(images[row * SAMPLES_PER_CLASS + col, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(str(row), rotation=0, labelpad=12)
    fig.suptitle(f"Requested digit by row — epoch {epoch}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

show_class_grid(0)  # Untrained baseline.

## 6. Train

Run for 30 epochs by default. Smoke mode runs one epoch on two batches. Each reported loss is an average weighted by batch size. Inspect sample grids for realism, correct labels, and within-class diversity; losses alone do not establish generation quality.

In [ ]:
epochs_to_run = 1 if SMOKE_TEST else EPOCHS
for epoch in range(1, epochs_to_run + 1):
    d_total = g_total = 0.0
    seen = 0
    for real_images, class_labels in train_loader:
        real_images = real_images.to(DEVICE, non_blocking=True)
        class_labels = class_labels.to(DEVICE, non_blocking=True)
        d_loss, g_loss = train_step(G, D, opt_G, opt_D, real_images, class_labels)
        count = real_images.size(0)
        d_total += d_loss * count
        g_total += g_loss * count
        seen += count
    d_losses.append(d_total / seen)
    g_losses.append(g_total / seen)
    print(f"Epoch {epoch}/{epochs_to_run} | D: {d_losses[-1]:.4f} | G: {g_losses[-1]:.4f}")
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs_to_run:
        show_class_grid(epoch)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(d_losses) + 1), d_losses, label="Discriminator")
ax.plot(range(1, len(g_losses) + 1), g_losses, label="Generator")
ax.set(xlabel="Epoch", ylabel="Loss", title="Conditional GAN training losses")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loss_curve.png")
plt.show()
plt.close(fig)

## 7. Request a particular digit

Change `REQUESTED_DIGIT` to any integer from 0 to 9. All images use that condition, while noise changes across examples. A smoke run will still produce essentially untrained samples.

In [ ]:
REQUESTED_DIGIT = 7
assert isinstance(REQUESTED_DIGIT, int) and 0 <= REQUESTED_DIGIT < NUM_CLASSES
noise = torch.randn(16, LATENT_DIM, device=DEVICE)
labels = torch.full((16,), REQUESTED_DIGIT, dtype=torch.long, device=DEVICE)
images = generate_images(noise, labels)
grid = make_grid(images, nrow=4)
plt.figure(figsize=(4, 4))
plt.imshow(grid.permute(1, 2, 0).numpy(), vmin=0, vmax=1)
plt.title(f"Requested digit: {REQUESTED_DIGIT}")
plt.axis("off")
plt.show()
save_image(images, OUTPUT_DIR / f"requested_digit_{REQUESTED_DIGIT}.png", nrow=4)

## 8. Save a checkpoint

Save model and optimizer states, configuration, and loss history. The optional reload snippet restores the generator for inference after its class has been defined. This checkpoint does not capture random-number-generator or data-loader state for exact training replay.

In [ ]:
checkpoint_path = OUTPUT_DIR / "conditional_gan_mnist.pt"
torch.save({
    "generator": G.state_dict(),
    "discriminator": D.state_dict(),
    "optimizer_g": opt_G.state_dict(),
    "optimizer_d": opt_D.state_dict(),
    "epochs_completed": len(g_losses),
    "config": {"latent_dim": LATENT_DIM, "num_classes": NUM_CLASSES,
               "img_size": IMG_SIZE, "seed": SEED, "smoke_test": SMOKE_TEST},
    "g_losses": g_losses,
    "d_losses": d_losses,
}, checkpoint_path)
print("Saved:", checkpoint_path.resolve())

# Optional inference reload (after defining Generator):
# checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
# config = checkpoint["config"]
# G = Generator(config["latent_dim"], config["num_classes"]).to(DEVICE)
# G.load_state_dict(checkpoint["generator"])
# G.eval()

## Exercises

1. Hold the label fixed and vary noise. Is there variety within the requested class?
2. Hold noise fixed and vary the label. Does digit identity follow the condition?
3. Explain why the generator step uses adversarial targets of one for every digit class.
4. Explain why `detach()` appears only in the discriminator step.
5. Replace one-hot labels with learned embeddings and update the input dimensions.

Conditioning helps control output but does not remove mode collapse or unstable training. An independent MNIST classifier can measure class agreement; visual inspection remains useful for diversity and realism.